# M07 lab solution: chain rewrite

**Brief.** Rebuild one legacy chain as LCEL, add a branch that routes short and long inputs differently, and compare outputs.

The legacy chain is `LLMChain` from `langchain_classic`, the one notebook `04a` migrates. It answers a question from three Larkspur passages. The LCEL version is `prompt | llm | StrOutputParser()`. A `RunnableBranch` then sends inputs whose context is under a length threshold straight to the answer chain, and longer ones through a condense step first. All three run on 10 golden questions.

Key: `EXPERIENTIALLABS_API_KEY`.

In [ ]:
import sys
from pathlib import Path

from dotenv import load_dotenv

sys.path.insert(0, str(Path("../../../../src").resolve()))
load_dotenv()

DATA = Path("../data")

import statistics
import warnings

import pandas as pd
from langchain_classic.chains.llm import LLMChain
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableBranch, RunnablePassthrough

from helpers.utils import get_experientiallabs_llm

In [ ]:
from lab_kit import load_handbook, load_jsonl, tfidf_retriever
from langchain_text_splitters import RecursiveCharacterTextSplitter

pages = load_handbook(DATA / "larkspur" / "handbook")
golden = load_jsonl(DATA / "larkspur" / "golden.jsonl")
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = [c for p in pages for c in splitter.split_text(p["text"])]
print(len(pages), "handbook files,", len(chunks), "chunks,", len(golden), "golden questions")

retrieve = tfidf_retriever(chunks, k=3)
questions = [g for g in golden if g["evidence_spans"]][:10]
inputs = [{"id": g["id"], "question": g["question"], "context": "\n\n".join(retrieve(g["question"]))}
          for g in questions]
print([len(x["context"]) for x in inputs])

## 1. The legacy chain

`LLMChain` takes a dict and returns a dict with the reply under `text`. It still runs from `langchain_classic`, with a deprecation warning.

In [ ]:
llm = get_experientiallabs_llm()

ANSWER = PromptTemplate.from_template(
    "Answer the question using only the context. If the context does not answer it, say you don't know.\n\n"
    "Context:\n{context}\n\nQuestion: {question}\nAnswer:")

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    legacy_chain = LLMChain(llm=llm, prompt=ANSWER)

def run_legacy(x: dict) -> str:
    return legacy_chain.invoke({"context": x["context"], "question": x["question"]})["text"].strip()

## 2. The same chain in LCEL

Same prompt and model; the output parser turns the `AIMessage` into a string, so the output shape matches the legacy `text`.

In [ ]:
lcel_chain = ANSWER | llm | StrOutputParser()

def run_lcel(x: dict) -> str:
    return lcel_chain.invoke({"context": x["context"], "question": x["question"]}).strip()

## 3. A branch on input length

The threshold is the median context length of the 10 inputs, so both routes are used. Short contexts go straight to `lcel_chain`. Long ones go through a condense step that keeps only the sentences relevant to the question, then the same answer chain. `RunnableBranch` runs the first condition that is true, and the last argument is the default.

In [ ]:
THRESHOLD = statistics.median(len(x["context"]) for x in inputs)
print("threshold:", THRESHOLD, "characters")

CONDENSE = PromptTemplate.from_template(
    "Copy only the sentences from the context that help answer the question. Do not add anything.\n\n"
    "Context:\n{context}\n\nQuestion: {question}\nSentences:")

condense_then_answer = (
    RunnablePassthrough.assign(context=CONDENSE | llm | StrOutputParser())
    | lcel_chain
)

branch = RunnableBranch(
    (lambda x: len(x["context"]) <= THRESHOLD, lcel_chain),
    condense_then_answer,
)

def route_of(x: dict) -> str:
    return "short: answer" if len(x["context"]) <= THRESHOLD else "long: condense, then answer"

## 4. Run all three on 10 questions

In [ ]:
rows = []
for x in inputs:
    rows.append({"id": x["id"], "question": x["question"], "context chars": len(x["context"]),
                 "route": route_of(x), "legacy LLMChain": run_legacy(x), "LCEL": run_lcel(x),
                 "RunnableBranch": branch.invoke({"context": x["context"], "question": x["question"]}).strip()})
    print(x["id"], "done")

## Results

In [ ]:
side_by_side = pd.DataFrame(rows)
side_by_side["legacy == LCEL"] = side_by_side["legacy LLMChain"] == side_by_side["LCEL"]
pd.set_option("display.max_colwidth", 200)
side_by_side

In [ ]:
side_by_side.groupby("route").agg(questions=("id", "size"), identical_legacy_lcel=("legacy == LCEL", "sum"))

## What this shows

- The legacy chain and its LCEL rewrite send the same prompt to the same model, so their answers should match closely; exact string equality at temperature 0 is the strictest check, and a difference is worth reading, not assuming.
- The branch decides on a property of the input that code can measure, so the route of every question is known before any model call. Log it with the answer.
- The condense step costs one extra call on long inputs. Whether it changes the answers is visible in the `RunnableBranch` column next to `LCEL`.